# Vulcan training-patch artifact audit

Investigates whether artifacts are present **as supervised nuclear positives in the
effective training patches** of run
`Vulcan_2.5.4_hardneg_hn2392da6c18e0_iezqw_a0cf3b2a13`, and lets each flagged region
be classified and — only on an explicit commit — corrected.

This is an investigation. It does **not** assume contaminated labels are the cause of
the downstream artifacts. `ARTIFACT_TRACE_T0_Z6_L212.md` already showed that for
`T0_Z6_L212` the negative supervision is present and correct and the model still
predicts nuclear there, so contaminated positives are not a demonstrated explanation
for that example. Auditing the rest of the pool is still worthwhile.

## What this notebook will not do

No training, no inference, no background jobs. It does not change the segmentation
run, rebuild any pool, or touch the existing 185-patch gold-correction pool or the
124-patch reviewed hard-negative pool. Validation and the temporal holdout are never
corrected and never extended. Corrections require an explicit commit, and the commit
writes a **new** versioned directory of sparse overlays — original files are never
modified. Preparing a new model and launching training are separate, deliberate steps
that live elsewhere.

## What "effective" means

The audited labels are reconstructed in the production order:

    original gold pool rows
      -> gold_corrections overrides applied by stem   (185 patches)
      -> the run's own p3_split
      -> reviewed hard-negative rows appended to TRAINING only   (124 patches)

Split sizes are checked against `phase3_run.json` and the reconstruction aborts on any
mismatch, so what you review is what the model actually saw. Original and effective
labels are always shown side by side: a positive that the existing corrections already
removed is a historical finding, not current contamination.

## Signals are priorities, not labels

Every nomination signal is a reason to look. T0, weak NPC support, small size,
non-roundness and multinuclear droplets cannot by themselves prove that a label is an
artifact. A stratified random sample of unflagged positives **and** of supervised
negatives is always included, because a flag-only list cannot measure prevalence.

## Fresh-kernel recovery

Each cell below is self-contained. After a kernel restart, run the bootstrap cell and
then the audit cell — nothing from the training or segmentation notebook is needed. The
notebook's authoritative definitions (`PipelineConfig`, the pool readers, `p3_split`,
the packed loader, the augmentation) are loaded straight out of
`vulcan_training_2_5_3.ipynb` by AST filter: only definitions and constant assignments
are compiled, so no generation, training or export cell can fire as a side effect.

## 0. Bootstrap

Puts the audit modules on the path and reloads them, so an edit to a helper takes
effect without restarting. Writes nothing.

In [ ]:
from pathlib import Path
import importlib, sys

AUDIT_CODE_DIR = Path('/home/tdeibert/Projects/Nuclear_Scaling/Code_Repository/Python/Model_Training')
if str(AUDIT_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(AUDIT_CODE_DIR))
importlib.invalidate_caches()

import reviewed_hard_negatives, gold_artifact_corrections
import vulcan_audit_recovery, training_patch_audit, training_patch_resolver
for _module in (reviewed_hard_negatives, gold_artifact_corrections, vulcan_audit_recovery,
                training_patch_audit, training_patch_resolver):
    importlib.reload(_module)

import training_patch_audit as audit
import training_patch_resolver as resolver

print('audit modules loaded from', AUDIT_CODE_DIR)
print('run      :', vulcan_audit_recovery.DEFAULT_RUN_DIR.name)
print('commit   :', vulcan_audit_recovery.DEFAULT_COMMIT_DIR.name)

## 1. Audit / setup — one contained cell

Loads the effective training data, validates provenance, summarises supervision,
nominates candidates, and opens (or resumes) a review.

**Settings.** `AUDIT_TIMEPOINTS = None` censuses the whole pool. The full pass reads the
labels and weights of every effective patch (~12 MB each, tens of thousands of patches),
so expect tens of minutes on the first run; progress bars show rate and elapsed time per
shard. It is **cached and resumable per (source, timepoint)**, so an interrupted run
keeps the shards it finished and a second run is fast.

For a quick first pass that confirms provenance and the plumbing before committing to
the full read, set `AUDIT_TIMEPOINTS = (0,)`. Prevalence figures from a partial census
only describe the timepoints actually read.

In [ ]:
# One contained audit/setup cell. Read-only apart from the resumable census cache
# and the review state. Safe to re-run; safe after a kernel restart.
# The run and commit being audited. Stated explicitly so the notebook records
# which artifacts it describes, rather than depending on a module default.
AUDIT_RUN_DIR = vulcan_audit_recovery.DEFAULT_RUN_DIR
AUDIT_COMMIT_DIR = vulcan_audit_recovery.DEFAULT_COMMIT_DIR

AUDIT_TIMEPOINTS = None        # None = every timepoint; e.g. (0,) for a fast first pass
AUDIT_WORKERS = 8              # loky workers reading label/weight files
AUDIT_SETTINGS = audit.NominationSettings(
    sample_per_timepoint=20,            # stratified random unflagged positives
    negative_controls_per_timepoint=6,  # stratified supervised-negative controls
    max_candidates=None,                # cap the flagged shortlist if it is huge
)

SESSION = audit.run_audit(AUDIT_RUN_DIR, AUDIT_COMMIT_DIR,
                          workers=AUDIT_WORKERS, timepoints=AUDIT_TIMEPOINTS,
                          settings=AUDIT_SETTINGS)

from IPython.display import display
SUMMARY = SESSION.summary()
print('\n=== effective supervision by split ===')
display(SUMMARY['by_split'])
print('=== supervised nuclear-positive regions by timepoint ===')
display(SUMMARY['by_timepoint'])
print('=== patches by source and sample class ===')
display(SUMMARY['by_class'])

print('\n=== integrity problems (empty is good) ===')
_broken = SESSION.patches[SESSION.patches.problems.astype(str).str.len() > 0]
display(_broken[['stem', 'split', 'problems']] if len(_broken) else 'none')

print('=== candidates nominated for review ===')
display(SESSION.candidates[['region_id', 'kind', 't', 'z', 'area_um2',
                            'n_correctable_patches', 'priority_score', 'flags']].head(40))

REVIEW = resolver.ReviewState(SESSION)
print('\nreview progress:', REVIEW.progress())
print('review folder  :', REVIEW.root)

### What the flags mean

| flag | meaning |
|---|---|
| `confirmed_artifact_overlap` | overlaps a mask already confirmed as an artifact in the committed review |
| `negative_supervision_conflict` | another **effective training crop** supervises these same pixels as *not* nuclear |
| `ignored_stored_positive` | a stored positive left unweighted in some crop — the hard-negative builder guards on stored positives, so this has different consequences from a weighted conflict |
| `below_z_floor` | centre plane below the nuclear Z floor (6 inclusive) |
| `droplet_filling` | the positive covers an implausible fraction of its droplet |
| `wall_contact` | touches supervised droplet wall / edge labels |
| `z_isolated` | no supervised positive in either neighbouring Z plane |
| `seam_proximity` | centre lies near a mosaic stitching seam |
| `no_npc_label` | no supervised NPC label anywhere in the region |
| `odd_shape` | low solidity, or more than one hole |
| `tiny` | smaller than the gold ROI minimum nuclear area |
| `out_of_image` | includes padded, out-of-image pixels |
| `weak_nls_enrichment` | NLS median inside barely exceeds the droplet surround |
| `weak_npc_support` | no NPC enrichment in the annulus straddling the envelope |

`audit.SIGNAL_TIERS` records which tier each column came from. Label-derived signals
cover the whole census; the two image-derived signals (`nls_enrichment`,
`npc_rim_support`) are measured for the review set only, from the **stored, globally
normalized** input tensors — never from an independently contrast-stretched picture.
A region split across crops is left unmeasured rather than stitched together.

`negative_supervision_px` is `-1` where the exact cross-crop scan did not run.

## 2. Decide and resolve — the interactive cell

Inspect each candidate, classify it, and autosave. Decisions persist in
`REVIEW.root`; re-running this cell or restarting the kernel **resumes** rather than
erasing. Then commit explicitly.

| decision | effect |
|---|---|
| **keep real nuclear positive** | nothing is written. Restoring or adding a positive needs reviewed boundaries and consistent edge/equatorial/Z targets, which a button cannot supply |
| **confirmed artifact negative** | the *entire* cyan region is non-nuclear. Nuclear heads 4/5/6 get target 0 and an explicit weight over exactly those pixels; the Z-offset there is de-supervised so a masked-out regression never becomes an invented target |
| **mixed / needs local redraw** | recorded and left **unresolved**. Local partial redraw is not implemented, so no correction is written |
| **uncertain** | reviewed, no correction |
| **skip** | reviewed, no correction |

An unreviewed candidate is never treated as real. "Unknown mechanism" in the failure
field does not weaken a confident exact-mask artifact decision — record it and move on.
Adjacent-plane views are context only: a centre-plane decision is never transferred to a
neighbouring Z plane.

**Commit** requires every candidate to have a decision and a reviewer (`uncertain` and
`skip` are valid completed reviews), and applies changes only to explicitly approved
exact regions. Use *Preview commit* first. An approved region is propagated to **every**
eligible training crop it touches, so no crop is left contradicting another; crops in
the spatial or temporal holdout are reported in `withheld_regions.csv` and left
byte-identical. Contradictory overlapping decisions abort the commit.

In [ ]:
# Interactive decision/resolver cell. Autosaves; restarting resumes.
if 'SESSION' not in globals() or 'REVIEW' not in globals():
    raise RuntimeError('Run the audit cell above first.')
REVIEW = resolver.decision_cell(SESSION, REVIEW)

### Commit from code instead of the button

Equivalent to *Preview commit* / *COMMIT decisions*, for when the widget is awkward
over a remote connection. `commit` is idempotent: a completed commit recorded in the
review state is returned rather than written twice.

In [ ]:
from IPython.display import display

PLAN = resolver.preview_commit(SESSION, REVIEW)
display(PLAN['patches'])
if len(PLAN['withheld']):
    print('approved but withheld (no eligible training crop):')
    display(PLAN['withheld'])
if len(PLAN['contradictions']):
    print('CONTRADICTIONS — a commit will abort:')
    display(PLAN['contradictions'])

# Uncomment to commit. Writes a NEW versioned directory; nothing existing is modified.
# COMMITTED = resolver.commit(SESSION, REVIEW)
# print(COMMITTED)

## 3. Augmentation inspection — read-only, after the static labels

Reproduces the training augmentation on real patches with a fixed seed and checks what
augmentation can actually break: that no `UNANNOTATED` sentinel is blended into a real
value, that pixels mapped in from outside the patch lose their weight instead of
inventing supervision, and that the input and the packed target stay aligned.

This measures the transform. It does **not** claim that zoom or defocus augmentation
caused any particular artifact.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

AUG = audit.augmentation_check(SESSION, seed=11, count=3)
display(AUG['table'])

# Before / after, for the first example: input, mask target, and weight.
_example = AUG['examples'][0]
_layout = AUG['layout']
_n_y, _nuc = _layout['n_y'], _layout['nuc_col']
_fig, _axes = plt.subplots(2, 3, figsize=(15, 9))
for _row, (_x, _packed), _label in ((0, _example['before'], 'before'),
                                    (1, _example['after'], 'after')):
    _axes[_row, 0].imshow(_x[..., 6], cmap='gray', vmin=0, vmax=1)
    _axes[_row, 0].set_title(f'{_label}: normalized NLS (centre plane, channel 6)', fontsize=9)
    _axes[_row, 1].imshow(_packed[..., _nuc], vmin=0, vmax=1)
    _axes[_row, 1].set_title(f'{_label}: nucleus_interior target', fontsize=9)
    _axes[_row, 2].imshow(_packed[..., _n_y + _nuc], vmin=0)
    _axes[_row, 2].set_title(f'{_label}: nucleus_interior weight', fontsize=9)
for _axis in _axes.flat:
    _axis.set_xticks([]); _axis.set_yticks([])
_fig.suptitle(f'{_example["stem"]} — augmentation with seed {_example["seed"]}')
_fig.tight_layout(); plt.show(); plt.close(_fig)

## 4. Self-check

Runs the synthetic end-to-end suite: it builds a miniature but structurally real pool
(raw hyperstack, complete Phase 2 gold pool, a committed review with a gold-correction
pool and a reviewed hard-negative pool, and a run directory whose `config.json` hashes
to its own name), then drives the production code over it and prints PASS/FAIL.

Nothing here touches the real data; everything happens in a temporary directory.
Run it after editing any audit helper.

In [ ]:
import importlib
import test_training_patch_audit
importlib.reload(test_training_patch_audit)
SELFCHECK = test_training_patch_audit.run_checks()

## 5. After the commit

The commit directory contains:

| file | contents |
|---|---|
| `manifest.json` | staging → complete status, immutable source hashes, per-patch records, withheld regions |
| `labels/`, `weights/` | sparse overlays composed on top of the current effective labels |
| `masks/` | the exact approved region masks |
| `review/` | the decisions, the candidate snapshot, the autosave history, and the full region census |
| `affected_patches.csv` | every overlaid crop with corrected and removed-positive pixel counts |
| `withheld_regions.csv` | approved regions with no eligible training crop |
| `holdout_only_regions.csv` | supervised positives that exist only in validation or the temporal holdout |

Enabling an overlay for training is deliberately **not** wired up here.
`resolver.audit_corrected_rows(commit_dir, cfg, rows)` is the reader a later run would
use; it re-verifies both the effective-label digests and the overlay digests and
refuses holdout rows. Any such run needs a **new model identity** and new versioned
outputs, and must keep this commit's provenance chain with it.

If `holdout_only_regions.csv` contains annotations that look wrong, quarantine and
report them separately. Do not tune heldout labels to the model and keep calling the
result independent validation.